# Diavision – See Your Health. Predict Your Future.
Run cell 1 (install), then cell 2 (build & launch). Open the public Gradio link printed at the end.

In [1]:
!pip install -q "gradio>=5,<6" xgboost lightgbm catboost shap reportlab plotly scikit-learn pandas numpy matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 MB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.6/325.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 462.4/462.4 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 2.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-genai 2.12.1 requires pydantic<3.0.0,>=2.12.5, but you have pydantic 2.12.3 which is incompatible.
hf-gradio 0

In [2]:
# =============================================================================
#  Diavision – See Your Health. Predict Your Future.
#  Powered by Artificial Intelligence for Early Diabetes Detection and Smarter Healthcare.
#
#  Google Colab: run these two lines in the first cell, then run this file's code.
#      !pip install -q "gradio>=5,<6" xgboost lightgbm catboost shap reportlab plotly scikit-learn pandas numpy matplotlib
#      %run diavision.py        (or paste the whole file into a cell)
#
#  IMPORTANT: This is a screening/education tool, not a medical device. The
#  training data is SYNTHETIC, so all accuracy numbers are measured on that
#  generated dataset and are NOT real-world clinical performance.
#  To use real data, put a CSV named "diabetes_data.csv" next to this file with
#  columns: Age,BMI,Glucose,HbA1c,BloodPressure,Insulin,FamilyHistory,ActivityHrs,SleepHrs,Outcome
# =============================================================================
import os, io, re, json, math, time, sqlite3, warnings, datetime as dt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import gradio as gr
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.inspection import permutation_importance
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix)

warnings.filterwarnings("ignore")
SEED = 42

# Optional boosters -----------------------------------------------------------
try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None
try:
    from lightgbm import LGBMClassifier
except Exception:
    LGBMClassifier = None
try:
    from catboost import CatBoostClassifier
except Exception:
    CatBoostClassifier = None
try:
    import shap
except Exception:
    shap = None
try:
    from reportlab.lib.pagesizes import A4
    from reportlab.lib import colors
    from reportlab.lib.styles import getSampleStyleSheet
    from reportlab.lib.units import mm
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
except Exception:
    SimpleDocTemplate = None

# =============================================================================
# 1. DATA
# =============================================================================
FEATURES = ["Age", "BMI", "Glucose", "HbA1c", "BloodPressure", "Insulin",
            "FamilyHistory", "ActivityHrs", "SleepHrs"]
FACTOR_TEXT = {  # (raises risk, lowers risk)
    "Age": ("Older age", "Younger age"),
    "BMI": ("High BMI", "Healthy BMI"),
    "Glucose": ("High glucose level", "Normal glucose level"),
    "HbA1c": ("High HbA1c", "Normal HbA1c"),
    "BloodPressure": ("Elevated blood pressure", "Healthy blood pressure"),
    "Insulin": ("Abnormal insulin level", "Typical insulin level"),
    "FamilyHistory": ("Family history of diabetes", "No family history"),
    "ActivityHrs": ("Low physical activity", "Good physical activity"),
    "SleepHrs": ("Poor sleep", "Healthy sleep"),
}


def make_dataset(n=6000, seed=SEED):
    rng = np.random.default_rng(seed)
    age = np.clip(rng.normal(46, 15, n), 18, 90)
    fam = rng.binomial(1, 0.3, n)
    act = np.clip(rng.gamma(2.2, 1.6, n), 0, 20)
    sleep = np.clip(rng.normal(6.9, 1.2, n), 3, 11)
    bmi = np.clip(26 + 0.03 * (age - 46) + 0.8 * fam - 0.15 * act + rng.normal(0, 4.2, n), 16, 55)
    glu = np.clip(92 + 0.3 * (age - 46) + 1.4 * (bmi - 26) + 6 * fam - 0.8 * act + rng.normal(0, 20, n), 60, 320)
    a1c = np.clip(5.2 + 0.014 * (glu - 92) + 0.01 * (bmi - 26) + rng.normal(0, 0.4, n), 4, 14)
    bp = np.clip(114 + 0.4 * (age - 46) + 0.8 * (bmi - 26) + rng.normal(0, 11, n), 80, 200)
    ins = np.clip(np.exp(rng.normal(4.0 + 0.03 * (bmi - 26), 0.5, n)), 2, 600)
    z = (0.02 * (age - 46) + 0.07 * (bmi - 26) + 0.03 * (glu - 100) + 0.8 * (a1c - 5.6)
         + 0.008 * (bp - 120) + 0.0015 * (ins - 55) + 0.6 * fam - 0.07 * act + 0.06 * (sleep - 7.5) ** 2)
    z = (z - np.quantile(z, 0.70)) * 1.8
    y = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)
    df = pd.DataFrame({"Age": age.round(0), "BMI": bmi.round(1), "Glucose": glu.round(0), "HbA1c": a1c.round(1),
                       "BloodPressure": bp.round(0), "Insulin": ins.round(0), "FamilyHistory": fam,
                       "ActivityHrs": act.round(1), "SleepHrs": sleep.round(1), "Outcome": y})
    return df


def load_data():
    p = "diabetes_data.csv"
    if os.path.exists(p):
        d = pd.read_csv(p)
        if set(FEATURES + ["Outcome"]).issubset(d.columns):
            return d[FEATURES + ["Outcome"]].dropna(), "Uploaded CSV (diabetes_data.csv)"
    return make_dataset(), "Synthetic (generated)"


DF, DATA_SOURCE = load_data()
X, Y = DF[FEATURES], DF["Outcome"]
X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, Y, test_size=0.30, stratify=Y, random_state=SEED)
X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)
REF = X_tr.median()

# =============================================================================
# 2. MODEL TRAINING, COMPARISON & AUTOMATIC SELECTION
# =============================================================================
def get_models():
    m = {
        "Logistic Regression": Pipeline([("sc", StandardScaler()), ("m", LogisticRegression(max_iter=2000))]),
        "Decision Tree": DecisionTreeClassifier(max_depth=5, min_samples_leaf=20, random_state=SEED),
        "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=SEED),
        "SVM": Pipeline([("sc", StandardScaler()), ("m", SVC(probability=True, random_state=SEED))]),
    }
    if XGBClassifier:
        m["XGBoost"] = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.9,
                                     colsample_bytree=0.9, eval_metric="logloss", random_state=SEED, n_jobs=-1)
    if LGBMClassifier:
        m["LightGBM"] = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=15, random_state=SEED, verbose=-1)
    if CatBoostClassifier:
        m["CatBoost"] = CatBoostClassifier(iterations=300, depth=4, learning_rate=0.05, verbose=0, random_seed=SEED)
    return m


def metrics(model, Xs, ys):
    p = model.predict_proba(Xs)[:, 1]
    yh = (p >= 0.5).astype(int)
    return {"Accuracy": accuracy_score(ys, yh), "Precision": precision_score(ys, yh, zero_division=0),
            "Recall": recall_score(ys, yh), "F1-Score": f1_score(ys, yh), "ROC-AUC": roc_auc_score(ys, p)}


print("Training models …")
TRAINED, ROWS = {}, []
for name, mdl in get_models().items():
    t0 = time.perf_counter()
    mdl.fit(X_tr, y_tr)
    v, t = metrics(mdl, X_val, y_val), metrics(mdl, X_te, y_te)
    TRAINED[name] = mdl
    ROWS.append({"Model": name, **{f"Val {k}": v[k] for k in v}, "Test Accuracy": t["Accuracy"], "Test ROC-AUC": t["ROC-AUC"]})
    print(f"  {name:<20} val acc {v['Accuracy']:.3f}  val auc {v['ROC-AUC']:.3f}  ({time.perf_counter()-t0:.1f}s)")

RES = pd.DataFrame(ROWS).sort_values(["Val ROC-AUC", "Val F1-Score"], ascending=False).reset_index(drop=True)
BEST_NAME = RES.loc[0, "Model"]            # AI Model Selection Engine: highest validation ROC-AUC (F1 breaks ties)
MODEL = TRAINED[BEST_NAME]
BEST_ROW = RES.loc[0]
P_TEST = MODEL.predict_proba(X_te)[:, 1]
MEAN_CONF = float(np.mean(np.maximum(P_TEST, 1 - P_TEST)))
print(f"Best model: {BEST_NAME}  | test acc {BEST_ROW['Test Accuracy']:.3f} | mean confidence {MEAN_CONF:.3f}")


def comparison_table():
    t = RES.copy()
    t.insert(0, "Rank", ["🏆" if i == 0 else str(i + 1) for i in range(len(t))])
    for c in t.columns[2:]:
        t[c] = (t[c] * 100).round(1).astype(str) + "%" if "AUC" not in c else t[c].round(3)
    return t


# =============================================================================
# 3. EXPLAINABILITY & CLINICAL HELPERS
# =============================================================================
def risk_category(p):
    r = p * 100
    if r <= 30: return "SAFE", "#2E9E6B", "🟢"
    if r <= 70: return "MODERATE RISK", "#E9A820", "🟡"
    return "HIGH RISK", "#E4572E", "🔴"


def local_factors(x):
    """Occlusion-style local explanation: how much does risk drop if a feature is set to the population median?"""
    base = float(MODEL.predict_proba(x)[0, 1])
    variants = []
    for f in FEATURES:
        v = x.copy(); v[f] = REF[f]; variants.append(v)
    c = base - MODEL.predict_proba(pd.concat(variants, ignore_index=True))[:, 1]
    tot = np.abs(c).sum() or 1.0
    out = [(f, float(ci), 100 * abs(ci) / tot) for f, ci in zip(FEATURES, c)]
    return sorted(out, key=lambda r: -r[2])


def health_score(p, bmi, act, sleep):
    s = 25 + 75 * (1 - p)
    s -= 8 if bmi >= 30 else 4 if (bmi >= 25 or bmi < 18.5) else 0
    s -= 5 if act < 2.5 else 0
    s -= 4 if (sleep < 6 or sleep > 9) else 0
    s = int(max(0, min(100, round(s))))
    return s, ("Excellent" if s >= 80 else "Moderate" if s >= 50 else "Needs Attention")


def bmi_calc(w, h_cm):
    return w / ((h_cm / 100) ** 2)


def ideal_weight(h_cm, gender):  # Devine formula
    inches = h_cm / 2.54
    return (50 if gender == "Male" else 45.5) + 2.3 * max(inches - 60, 0)


def water_litres(w, act):
    return round(w * 0.035 + (0.5 if act >= 3 else 0), 1)


def calorie_calc(w, h, age, gender, act):
    bmr = 10 * w + 6.25 * h - 5 * age + (5 if gender == "Male" else -161)
    f = 1.2 if act < 1 else 1.375 if act < 3 else 1.55 if act < 5 else 1.725 if act < 8 else 1.9
    return round(bmr), round(bmr * f)


def build_recs(d, p):
    bmi, w, h = d["BMI"], d["Weight"], d["Height"]
    bmr, tdee = calorie_calc(w, h, d["Age"], d["Gender"], d["ActivityHrs"])
    floor = 1500 if d["Gender"] == "Male" else 1200
    kcal = max(floor, tdee - 400) if bmi >= 25 else tdee + 300 if bmi < 18.5 else tdee
    kcal = int(round(kcal / 50) * 50)
    protein = int(round(0.9 * w)); fiber = max(25, int(round(kcal / 1000 * 14)))
    water = water_litres(w, d["ActivityHrs"])
    steps = 6000 if d["Age"] >= 65 else 7000 if d["ActivityHrs"] < 2 else 9000
    # sleep
    target = 8.0 if d["SleepHrs"] < 6.5 else 7.5
    wh, wm = map(int, d["Wake"].split(":"))
    bed = (dt.datetime(2000, 1, 1, wh, wm) - dt.timedelta(hours=target + 0.25)).strftime("%I:%M %p")
    sleep_tips = ["Keep the same bed and wake time every day, including weekends.",
                  "Stop caffeine after 2 pm and keep dinner at least 2–3 hours before bed.",
                  "Keep the bedroom dark, cool and screen-free."]
    if d["SleepHrs"] < 6: sleep_tips.insert(0, "You sleep under 6 hours – add 30 minutes every week until you reach the target.")
    meals = [("Mon", "Moong dal chilla + mint chutney", "Brown rice, rajma, cucumber salad", "Palak paneer, 1 multigrain roti"),
             ("Tue", "Vegetable oats upma", "Jowar roti, mixed-veg sabzi, curd", "Lentil soup, tofu-vegetable stir-fry"),
             ("Wed", "Besan-methi thepla (small) + curd", "Quinoa-chickpea bowl", "Lauki sabzi, dal, 1 roti"),
             ("Thu", "Ragi dosa + sambar", "Brown rice, kadhi, bhindi", "Paneer tikka, grilled veg, salad"),
             ("Fri", "Sprouts poha with vegetables", "Chana masala, 2 bajra roti, salad", "Dal-heavy veg khichdi + raita"),
             ("Sat", "2 idli + sambar", "Soya-chunk pulao + raita", "Tomato-lentil soup, stuffed capsicum"),
             ("Sun", "Vegetable besan cheela", "Brown rice, dal tadka, karela/lauki sabzi", "Millet-vegetable khichdi, buttermilk")]
    return {
        "kcal": kcal, "protein": protein, "fiber": fiber, "water": water, "meals": meals,
        "avoid": ["Sugary drinks and packaged fruit juices", "Sweets, mithai and desserts", "Maida, white bread, bakery items",
                  "Deep-fried snacks (samosa, pakora, chips)", "Large portions of white rice", "Highly processed packaged foods"],
        "snacks": "Roasted chana, a handful of nuts, guava or apple slices, unsweetened buttermilk.",
        "steps": steps, "weekly_min": 150 if d["Age"] < 65 else 120,
        "walk": "Take a 10–15 minute walk after each main meal; build up to 30 minutes of brisk walking on most days.",
        "yoga": ["Surya Namaskar – 5 slow rounds", "Vajrasana for 5 minutes after meals", "Ardha Matsyendrasana (seated twist)",
                 "Mandukasana and Paschimottanasana", "Pranayama (Anulom-Vilom) for 5 minutes"],
        "home": ["Chair squats – 3 × 12", "Wall push-ups – 3 × 10", "Glute bridges – 3 × 12", "Plank – 3 × 20–30 s",
                 "2 strength sessions per week"],
        "bed": bed, "wake": datetime_fmt(wh, wm), "sleep_h": target, "sleep_tips": sleep_tips,
        "stress": ["10 minutes of mindfulness meditation each morning", "Box breathing (4-4-4-4) or 4-7-8 breathing when stressed",
                   "No screens for the last hour before bed; limit recreational screen time to 2 hours",
                   f"Hydration reminder: a glass of water (~250 ml) every 2 hours – about {water} L per day"],
        "caution": "Check with your doctor before starting a new exercise plan." if (p > 0.5 or d["Age"] >= 50) else "",
    }


def datetime_fmt(h, m):
    return dt.datetime(2000, 1, 1, h, m).strftime("%I:%M %p")


def ai_recommendation(p, factors, d):
    cat = risk_category(p)[0]
    top = ", ".join(FACTOR_TEXT[f][0 if c > 0 else 1].lower() for f, c, _ in factors[:2] if c > 0) or "no single dominant factor"
    if cat == "HIGH RISK":
        return (f"Your profile shows a high likelihood of diabetes, driven mainly by {top}. Please see a doctor soon for a fasting "
                "plasma glucose / HbA1c confirmation test, and begin the nutrition and activity plan below.")
    if cat == "MODERATE RISK":
        return (f"Your risk is moderate, mainly due to {top}. Focus on weight, diet quality and activity, and repeat screening "
                "every 6–12 months.")
    return "Your risk is low. Keep up your healthy habits and screen yearly, especially if you have a family history."


# =============================================================================
# 4. PATIENT DATABASE (SQLite)
# =============================================================================
DB_PATH = "diavision.db"


def db():
    c = sqlite3.connect(DB_PATH, check_same_thread=False)
    c.execute("CREATE TABLE IF NOT EXISTS patients(id INTEGER PRIMARY KEY AUTOINCREMENT, pid TEXT UNIQUE, name TEXT, email TEXT)")
    c.execute("""CREATE TABLE IF NOT EXISTS predictions(id INTEGER PRIMARY KEY AUTOINCREMENT, pid TEXT, created TEXT, model TEXT,
                 probability REAL, confidence REAL, risk_level TEXT, health_score INTEGER, inputs TEXT)""")
    return c


def save_prediction(name, email, p, conf, risk, score, inputs):
    c = db()
    row = c.execute("SELECT pid FROM patients WHERE lower(email)=lower(?)", (email,)).fetchone() if email else None
    if row:
        pid = row[0]
    else:
        cur = c.execute("INSERT INTO patients(name,email) VALUES(?,?)", (name, email))
        pid = f"DV{cur.lastrowid:06d}"
        c.execute("UPDATE patients SET pid=? WHERE id=?", (pid, cur.lastrowid))
    now = dt.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    c.execute("INSERT INTO predictions(pid,created,model,probability,confidence,risk_level,health_score,inputs) VALUES(?,?,?,?,?,?,?,?)",
              (pid, now, BEST_NAME, p, conf, risk, score, json.dumps(inputs)))
    c.commit(); c.close()
    return pid, now


def history_df(q=""):
    c = db()
    sql = """SELECT pr.pid AS 'Patient ID', pa.name AS Name, pa.email AS Email, pr.created AS 'Date & Time', pr.model AS 'Model Used',
             ROUND(pr.probability*100,1) AS 'Risk %', ROUND(pr.confidence*100,1) AS 'Confidence %', pr.risk_level AS 'Risk Level',
             pr.health_score AS 'Health Score' FROM predictions pr JOIN patients pa ON pa.pid=pr.pid"""
    args = ()
    if q.strip():
        sql += " WHERE pr.pid LIKE ? OR pa.name LIKE ? OR pa.email LIKE ?"
        args = (f"%{q.strip()}%",) * 3
    df = pd.read_sql_query(sql + " ORDER BY pr.id DESC LIMIT 500", c, params=args)
    c.close()
    return df


def export_csv():
    path = "diavision_history.csv"
    history_df().to_csv(path, index=False)
    return path


# =============================================================================
# 5. HTML COMPONENTS (Bootstrap 5 classes + custom CSS)
# =============================================================================
def _pt(a, R=120):
    t = math.radians(a)
    return 150 + R * math.cos(t), 150 - R * math.sin(t)


def _arc(p0, p1, col):
    x0, y0 = _pt(180 - 1.8 * p0); x1, y1 = _pt(180 - 1.8 * p1)
    return f'<path d="M{x0:.1f} {y0:.1f} A120 120 0 0 1 {x1:.1f} {y1:.1f}" stroke="{col}" stroke-width="22" fill="none"/>'


def meter_html(p):
    cat, col, ico = risk_category(p)
    rot = -90 + 180 * p
    ticks = "".join(f'<text x="{_pt(180-1.8*v, 142)[0]:.0f}" y="{_pt(180-1.8*v, 142)[1]:.0f}" text-anchor="middle" '
                    f'font-size="10" fill="currentColor" opacity=".6">{v}</text>' for v in (0, 30, 70, 100))
    return f"""<div class="dv-card text-center"><div class="dv-k">AI Risk Meter</div>
<svg viewBox="0 0 300 200" style="max-width:480px;width:100%">{_arc(0,30,'#2E9E6B')}{_arc(30,70,'#E9A820')}{_arc(70,100,'#E4572E')}{ticks}
<g class="dv-needle" style="--to:{rot:.1f}deg"><line x1="150" y1="150" x2="150" y2="46" stroke="currentColor" stroke-width="4" stroke-linecap="round"/></g>
<circle cx="150" cy="150" r="9" fill="currentColor"/>
<text x="150" y="182" text-anchor="middle" font-size="26" font-weight="700" fill="{col}">{p*100:.1f}%</text></svg>
<div style="font-size:1.4rem;font-weight:700;color:{col}">{ico} {cat}</div>
<div class="dv-k">0–30% safe · 31–70% moderate · 71–100% high</div></div>"""


def result_html(r):
    cat, col, ico = risk_category(r["p"])
    cards = [("Patient ID", r["pid"]), ("AI Prediction", "✅ " + ("Diabetes likely" if r["p"] >= .5 else "Diabetes unlikely")),
             ("Confidence", f"{r['conf']*100:.1f}%"), ("Health Score", f"{r['score']}/100 · {r['score_label']}"),
             ("Risk Percentage", f"{r['p']*100:.1f}%"), ("Probability of Diabetes", f"{r['p']:.3f}"),
             ("Risk Category", f'<span style="color:{col}">{ico} {cat}</span>'), ("Model Used", r["model"]),
             ("Prediction Time", f"{r['time']} · {r['ms']:.0f} ms")]
    body = "".join(f'<div class="col-6 col-md-4"><div class="dv-card"><div class="dv-k">{k}</div><div class="dv-v" style="font-size:1.15rem">{v}</div></div></div>' for k, v in cards)
    return (f'<div class="row g-2">{body}</div><div class="dv-card mt-2"><div class="dv-k">AI Recommendation</div>'
            f'<div>{r["advice"]}</div></div>')


def xai_html(factors):
    rows = ""
    for f, c, pct in factors[:6]:
        up = c > 0
        lab = FACTOR_TEXT[f][0 if up else 1] + (" ↑" if up else " ↓")
        colr = "#E4572E" if up else "#2E9E6B"
        rows += (f'<div class="mb-3"><div class="d-flex justify-content-between"><span>{lab}</span><b>{pct:.1f}%</b></div>'
                 f'<div class="progress" style="height:12px"><div class="progress-bar dv-bar" style="width:{pct:.1f}%;background:{colr}"></div></div></div>')
    return (f'<div class="dv-card"><div class="dv-k mb-2">Explainable AI – top factors behind this prediction '
            f'(<span style="color:#E4572E">↑ raises risk</span>, <span style="color:#2E9E6B">↓ lowers risk</span>)</div>{rows}</div>')


def score_html(s, label):
    col = "#2E9E6B" if s >= 80 else "#E9A820" if s >= 50 else "#E4572E"
    return (f'<div class="dv-card"><div class="dv-k">AI Health Score</div><div class="dv-v" style="color:{col};font-size:2.2rem">{s}/100 '
            f'<span style="font-size:1.1rem">→ {label}</span></div><div class="progress" style="height:12px">'
            f'<div class="progress-bar dv-bar" style="width:{s}%;background:{col}"></div></div></div>')


def lst(items):
    return "<ul class='mb-0'>" + "".join(f"<li>{i}</li>" for i in items) + "</ul>"


def recs_html(R):
    meals = "".join(f"<tr><td><b>{d}</b></td><td>{a}</td><td>{b}</td><td>{c}</td></tr>" for d, a, b, c in R["meals"])
    n = (f"<div class='row g-2 mb-2'>" + "".join(f"<div class='col-6 col-md-3'><div class='dv-k'>{k}</div><b>{v}</b></div>" for k, v in
         [("Calories", f"{R['kcal']} kcal/day"), ("Protein", f"{R['protein']} g/day"), ("Fiber", f"{R['fiber']} g/day"), ("Water", f"{R['water']} L/day")]) + "</div>")
    nutri = (f"{n}<div class='dv-k'>Foods to avoid</div>{lst(R['avoid'])}<div class='dv-k mt-2'>Smart snacks</div><div>{R['snacks']}</div>"
             f"<div class='dv-k mt-2'>Weekly vegetarian meal suggestions</div><div style='overflow-x:auto'><table class='table table-sm'>"
             f"<thead><tr><th></th><th>Breakfast</th><th>Lunch</th><th>Dinner</th></tr></thead><tbody>{meals}</tbody></table></div>")
    ex = (f"<div class='mb-2'><b>Daily step goal:</b> {R['steps']:,} · <b>Weekly activity target:</b> {R['weekly_min']} min moderate + 2 strength sessions</div>"
          f"<div class='dv-k'>Walking plan</div><div class='mb-2'>{R['walk']}</div><div class='dv-k'>Yoga routine</div>{lst(R['yoga'])}"
          f"<div class='dv-k mt-2'>Home workout</div>{lst(R['home'])}" + (f"<div class='mt-2'><i>{R['caution']}</i></div>" if R["caution"] else ""))
    sl = (f"<div class='mb-2'><b>Ideal bedtime:</b> {R['bed']} · <b>Wake-up:</b> {R['wake']} · <b>Duration:</b> {R['sleep_h']} h</div>{lst(R['sleep_tips'])}")
    return f"""<div class="row g-2">
<div class="col-12 col-lg-6"><div class="dv-card"><h5>🥗 Nutrition</h5>{nutri}</div></div>
<div class="col-12 col-lg-6"><div class="dv-card"><h5>🏃 Exercise</h5>{ex}</div></div>
<div class="col-12 col-lg-6"><div class="dv-card"><h5>😴 Sleep</h5>{sl}</div></div>
<div class="col-12 col-lg-6"><div class="dv-card"><h5>🧘 Stress management</h5>{lst(R['stress'])}</div></div></div>"""


# =============================================================================
# 6. PREDICTION PIPELINE
# =============================================================================
def timeline_fig(x):
    ys = list(range(11)); cur, imp = [], []
    for t in ys:
        r = x.iloc[0].copy()
        r["Age"] += t; r["BMI"] += .12 * t; r["Glucose"] += t; r["HbA1c"] += .02 * t; r["BloodPressure"] += .5 * t
        r["ActivityHrs"] = max(r["ActivityHrs"] - .05 * t, 0); cur.append(r)
        s = x.iloc[0].copy(); k = min(t, 5) / 5
        s["Age"] += t
        s["BMI"] -= .6 * max(s["BMI"] - 23, 0) * k
        s["Glucose"] += .2 * t - .15 * max(s["Glucose"] - 95, 0) * k
        s["HbA1c"] -= .15 * max(s["HbA1c"] - 5.4, 0) * k
        s["BloodPressure"] -= .1 * max(s["BloodPressure"] - 120, 0) * k
        s["ActivityHrs"] += (max(s["ActivityHrs"], 5) - s["ActivityHrs"]) * k
        s["SleepHrs"] += (7.5 - s["SleepHrs"]) * k; imp.append(s)
    a = MODEL.predict_proba(pd.DataFrame(cur)[FEATURES].astype(float))[:, 1] * 100
    b = MODEL.predict_proba(pd.DataFrame(imp)[FEATURES].astype(float))[:, 1] * 100
    f = go.Figure()
    f.add_trace(go.Scatter(x=ys, y=a, name="No lifestyle change", line=dict(color="#E4572E", width=3)))
    f.add_trace(go.Scatter(x=ys, y=b, name="Following the AI plan", line=dict(color="#2E9E6B", width=3)))
    f.add_hrect(y0=0, y1=30, fillcolor="#2E9E6B", opacity=.07, line_width=0)
    f.add_hrect(y0=70, y1=100, fillcolor="#E4572E", opacity=.07, line_width=0)
    f.update_layout(title="Diabetes risk timeline – next 10 years (illustrative projection)", xaxis_title="Years from now",
                    yaxis_title="Risk %", yaxis_range=[0, 100], height=380, legend=dict(orientation="h", y=-.25),
                    margin=dict(l=40, r=20, t=50, b=40))
    return f


def run_prediction(name, email, age, gender, height, weight, glucose, hba1c, bp, insulin, family, activity, sleep, wake):
    if not name or not name.strip():
        raise gr.Error("Enter the patient's name.")
    if email and not re.match(r"[^@\s]+@[^@\s]+\.[^@\s]+$", email):
        raise gr.Error("Enter a valid email address or leave it blank.")
    for lab, v, lo, hi in [("Age", age, 18, 100), ("Height (cm)", height, 100, 230), ("Weight (kg)", weight, 30, 250),
                           ("Glucose", glucose, 40, 500), ("HbA1c", hba1c, 3, 16), ("Blood pressure", bp, 60, 240),
                           ("Insulin", insulin, 0, 900)]:
        if v is None or not (lo <= v <= hi):
            raise gr.Error(f"{lab} must be between {lo} and {hi}.")
    bmi = round(bmi_calc(weight, height), 1)
    d = dict(Name=name.strip(), Email=(email or "").strip(), Age=age, Gender=gender, Height=height, Weight=weight, BMI=bmi,
             Glucose=glucose, HbA1c=hba1c, BloodPressure=bp, Insulin=insulin, FamilyHistory=int(family == "Yes"),
             ActivityHrs=activity, SleepHrs=sleep, Wake=wake)
    x = pd.DataFrame([[d[f] for f in FEATURES]], columns=FEATURES).astype(float)
    t0 = time.perf_counter()
    p = float(MODEL.predict_proba(x)[0, 1])
    ms = (time.perf_counter() - t0) * 1000
    conf = max(p, 1 - p)
    factors = local_factors(x)
    score, slabel = health_score(p, bmi, activity, sleep)
    R = build_recs(d, p)
    advice = ai_recommendation(p, factors, d)
    pid, now = save_prediction(d["Name"], d["Email"], p, conf, risk_category(p)[0], score, d)
    r = dict(p=p, conf=conf, score=score, score_label=slabel, model=BEST_NAME, ms=ms, time=now, pid=pid, advice=advice,
             d=d, factors=[(f, c, pc) for f, c, pc in factors], R=R)
    return (result_html(r), meter_html(p), xai_html(factors), score_html(score, slabel), recs_html(R), timeline_fig(x), r)


# =============================================================================
# 7. PDF REPORT
# =============================================================================
def make_pdf(r):
    if r is None:
        raise gr.Error("Run a prediction first.")
    if SimpleDocTemplate is None:
        raise gr.Error("ReportLab is not installed: pip install reportlab")
    path = f"Diavision_Report_{r['pid']}.pdf"
    st = getSampleStyleSheet()
    doc = SimpleDocTemplate(path, pagesize=A4, leftMargin=18 * mm, rightMargin=18 * mm, topMargin=16 * mm, bottomMargin=16 * mm)
    d, R = r["d"], r["R"]
    cat = risk_category(r["p"])[0]

    def tbl(rows, widths=None):
        t = Table(rows, colWidths=widths)
        t.setStyle(TableStyle([("GRID", (0, 0), (-1, -1), .4, colors.HexColor("#C9D9D6")),
                               ("BACKGROUND", (0, 0), (0, -1), colors.HexColor("#EAF5F3")),
                               ("FONTSIZE", (0, 0), (-1, -1), 9), ("VALIGN", (0, 0), (-1, -1), "TOP")]))
        return t
    S = [Paragraph("Diavision – AI Diabetes Screening Report", st["Title"]),
         Paragraph("See Your Health. Predict Your Future.", st["Italic"]), Spacer(1, 8),
         Paragraph("Patient", st["Heading3"]),
         tbl([["Patient ID", r["pid"]], ["Name", d["Name"]], ["Email", d["Email"] or "-"], ["Age / Gender", f"{d['Age']:.0f} / {d['Gender']}"],
              ["Height / Weight / BMI", f"{d['Height']:.0f} cm / {d['Weight']:.0f} kg / {d['BMI']}"],
              ["Glucose / HbA1c", f"{d['Glucose']:.0f} mg/dL / {d['HbA1c']}%"], ["Blood pressure / Insulin", f"{d['BloodPressure']:.0f} / {d['Insulin']:.0f}"],
              ["Family history", "Yes" if d["FamilyHistory"] else "No"], ["Activity / Sleep", f"{d['ActivityHrs']} h/week / {d['SleepHrs']} h"]],
             [55 * mm, 115 * mm]),
         Spacer(1, 8), Paragraph("AI Prediction", st["Heading3"]),
         tbl([["Prediction", "Diabetes likely" if r["p"] >= .5 else "Diabetes unlikely"], ["Risk", f"{r['p']*100:.1f}% ({cat})"],
              ["Confidence", f"{r['conf']*100:.1f}%"], ["Health score", f"{r['score']}/100 ({r['score_label']})"],
              ["Model used", r["model"]], ["Date & time", r["time"]]], [55 * mm, 115 * mm]),
         Spacer(1, 8), Paragraph("Top factors", st["Heading3"]),
         tbl([[FACTOR_TEXT[f][0 if c > 0 else 1], ("raises risk" if c > 0 else "lowers risk"), f"{pc:.1f}%"] for f, c, pc in r["factors"][:6]],
             [80 * mm, 45 * mm, 45 * mm]),
         Spacer(1, 8), Paragraph("AI recommendation", st["Heading3"]), Paragraph(r["advice"], st["BodyText"]),
         Paragraph("Lifestyle plan", st["Heading3"]),
         Paragraph(f"<b>Nutrition:</b> {R['kcal']} kcal, {R['protein']} g protein, {R['fiber']} g fiber, {R['water']} L water per day (vegetarian, low-GI).", st["BodyText"]),
         Paragraph(f"<b>Exercise:</b> {R['steps']:,} steps/day; {R['weekly_min']} min moderate activity + 2 strength sessions per week.", st["BodyText"]),
         Paragraph(f"<b>Sleep:</b> bedtime {R['bed']}, wake {R['wake']}, {R['sleep_h']} h.", st["BodyText"]),
         Paragraph("<b>Stress:</b> " + "; ".join(R["stress"][:3]) + ".", st["BodyText"]), Spacer(1, 10),
         Paragraph("<i>Disclaimer: This report is an AI screening aid built on a model whose accuracy was measured on "
                   f"{DATA_SOURCE.lower()} data. It is not a diagnosis. Please consult a qualified doctor.</i>", st["BodyText"])]
    doc.build(S)
    return path


# =============================================================================
# 8. ANALYTICS DASHBOARD
# =============================================================================
TEAL = "#0E8A8A"; CORAL = "#E4572E"


def fig_cm():
    cm = confusion_matrix(y_te, (P_TEST >= .5).astype(int))
    f = px.imshow(cm, text_auto=True, x=["Non-diabetic", "Diabetic"], y=["Non-diabetic", "Diabetic"],
                  color_continuous_scale=["#EAF5F3", TEAL], labels=dict(x="Predicted", y="Actual"))
    return f.update_layout(title=f"Confusion matrix – {BEST_NAME} (test set)", height=380, coloraxis_showscale=False)


def fig_roc():
    f = go.Figure()
    for n, m in TRAINED.items():
        fpr, tpr, _ = roc_curve(y_te, m.predict_proba(X_te)[:, 1])
        f.add_trace(go.Scatter(x=fpr, y=tpr, name=f"{n} ({roc_auc_score(y_te, m.predict_proba(X_te)[:,1]):.3f})",
                               line=dict(width=4 if n == BEST_NAME else 1.5)))
    f.add_trace(go.Scatter(x=[0, 1], y=[0, 1], line=dict(dash="dash", color="grey"), showlegend=False))
    return f.update_layout(title="ROC curves (test set)", xaxis_title="False positive rate", yaxis_title="True positive rate", height=380)


PERM = None


def global_importance():
    global PERM
    if PERM is None:
        r = permutation_importance(MODEL, X_val, y_val, scoring="roc_auc", n_repeats=5, random_state=SEED)
        PERM = pd.Series(r.importances_mean, index=FEATURES).clip(lower=0).sort_values()
    return PERM


def fig_importance():
    s = global_importance(); s = s / (s.sum() or 1) * 100
    f = go.Figure(go.Bar(x=s.values, y=s.index, orientation="h", marker_color=TEAL))
    return f.update_layout(title="Feature importance (permutation, validation set)", xaxis_title="% of total importance", height=380)


def fig_shap():
    if shap is not None:
        try:
            bg = X_tr.sample(100, random_state=SEED); sm = X_te.sample(250, random_state=SEED)
            ex = shap.Explainer(lambda a: MODEL.predict_proba(pd.DataFrame(a, columns=FEATURES))[:, 1], bg)
            sv = ex(sm).values
            order = np.argsort(np.abs(sv).mean(0))
            f = go.Figure(); rng = np.random.default_rng(1)
            for rank, i in enumerate(order):
                col = sm.iloc[:, i]; nrm = (col - col.min()) / ((col.max() - col.min()) or 1)
                f.add_trace(go.Scatter(x=sv[:, i], y=rank + rng.uniform(-.3, .3, len(sv)), mode="markers", showlegend=False,
                                       marker=dict(size=5, color=nrm, colorscale=[[0, "#2F6DB5"], [1, CORAL]], opacity=.75),
                                       hovertext=[f"{FEATURES[i]} = {v}" for v in col]))
            f.update_layout(title="SHAP summary (red = high feature value)", height=380,
                            yaxis=dict(tickmode="array", tickvals=list(range(len(order))), ticktext=[FEATURES[i] for i in order]),
                            xaxis_title="SHAP value (impact on diabetes probability)")
            return f
        except Exception as e:
            print("SHAP unavailable, using feature importance:", e)
    return fig_importance().update_layout(title="Explainability (feature importance – SHAP unavailable)")


def fig_corr():
    c = DF.corr().round(2)
    return px.imshow(c, text_auto=True, color_continuous_scale="RdBu_r", zmin=-1, zmax=1, aspect="auto").update_layout(
        title="Correlation heatmap", height=520)


def fig_risk_dist():
    pr = MODEL.predict_proba(X)[:, 1] * 100
    cnt = pd.cut(pr, [-.1, 30, 70, 100], labels=["Safe (0–30%)", "Moderate (31–70%)", "High (71–100%)"]).value_counts().sort_index()
    return go.Figure(go.Bar(x=cnt.index.astype(str), y=cnt.values, marker_color=["#2E9E6B", "#E9A820", CORAL])).update_layout(
        title="Risk distribution (all records)", height=380, yaxis_title="Patients")


def fig_hist(col, title):
    d = DF.assign(Group=DF["Outcome"].map({0: "Non-diabetic", 1: "Diabetic"}))
    return px.histogram(d, x=col, color="Group", barmode="overlay", opacity=.65, nbins=30,
                        color_discrete_map={"Non-diabetic": TEAL, "Diabetic": CORAL}).update_layout(title=title, height=380)


def stats_table():
    g = DF.groupby("Outcome")[FEATURES].mean().round(1).T
    g.columns = ["Non-diabetic (mean)", "Diabetic (mean)"]
    n = DF["Outcome"].value_counts()
    g.loc["Patients (count)"] = [n.get(0, 0), n.get(1, 0)]
    return g.reset_index().rename(columns={"index": "Measure"})


def fig_trend():
    c = db()
    d = pd.read_sql_query("SELECT substr(created,1,7) AS month, COUNT(*) AS n, AVG(probability)*100 AS risk FROM predictions GROUP BY month ORDER BY month", c)
    c.close()
    f = go.Figure()
    if d.empty:
        f.add_annotation(text="No predictions saved yet – run a prediction to see trends", showarrow=False, font=dict(size=14))
    else:
        f.add_trace(go.Bar(x=d["month"], y=d["n"], name="Predictions", marker_color=TEAL))
        f.add_trace(go.Scatter(x=d["month"], y=d["risk"], name="Avg risk %", yaxis="y2", line=dict(color=CORAL, width=3)))
    return f.update_layout(title="Monthly prediction trends", height=380, yaxis=dict(title="Predictions"),
                           yaxis2=dict(title="Avg risk %", overlaying="y", side="right", range=[0, 100]), legend=dict(orientation="h", y=-.25))


def fig_model_bars():
    f = go.Figure()
    for m in ["Val Accuracy", "Val F1-Score", "Val ROC-AUC"]:
        f.add_trace(go.Bar(x=RES["Model"], y=RES[m], name=m.replace("Val ", "")))
    return f.update_layout(barmode="group", title="Validation metrics by model", height=380, yaxis_range=[0, 1], legend=dict(orientation="h", y=-.25))


# =============================================================================
# 9. AI HEALTH ASSISTANT (rule-based, runs offline – no external LLM)
# =============================================================================
KB = [(("glucose", "sugar"), "Fasting glucose under 100 mg/dL is normal, 100–125 is pre-diabetes and 126+ (confirmed twice) suggests diabetes. Cut sugary drinks, favour fibre-rich carbs and walk after meals."),
      (("hba1c", "a1c"), "HbA1c reflects your 2–3 month average glucose: below 5.7% is normal, 5.7–6.4% pre-diabetes, 6.5%+ diabetes range."),
      (("bmi", "weight", "obes"), "A BMI of 18.5–24.9 is the usual healthy range (23 or lower is a common target for South Asian adults). Losing even 5–7% of body weight markedly reduces diabetes risk."),
      (("diet", "food", "eat", "meal", "veg"), "Build plates around vegetables, dals, sprouts, millets, paneer/tofu and whole grains; avoid sweets, maida, fried snacks and sugary drinks. See the Nutrition plan on the Predict tab."),
      (("exercise", "walk", "workout", "yoga", "step"), "Aim for 150 minutes of moderate activity a week plus 2 strength sessions, and a 10–15 minute walk after meals."),
      (("sleep", "insomnia"), "Adults need 7–9 hours. Short or irregular sleep raises insulin resistance. Keep fixed sleep times and avoid screens an hour before bed."),
      (("stress", "anxiety", "relax", "meditat"), "Chronic stress raises blood sugar. Try 10 minutes of meditation daily and slow breathing (4-7-8) during stressful moments."),
      (("symptom", "thirst", "urinat", "tired"), "Classic warning signs: excess thirst, frequent urination, unexplained weight loss, fatigue, blurred vision, slow-healing wounds. If you notice them, get tested promptly."),
      (("water", "hydrat"), "A rough target is 35 ml of water per kg body weight per day, more if you are active or it is hot."),
      (("doctor", "test", "diagnos"), "Only a doctor can diagnose diabetes (fasting glucose, HbA1c or an OGTT). Diavision is a screening aid, not a diagnosis.")]


def assistant(msg, hist, r):
    m = msg.lower(); reply = None
    if r and any(k in m for k in ("my risk", "my result", "my score", "my prediction", "summary")):
        reply = (f"Your last result ({r['d']['Name']}): risk {r['p']*100:.1f}% ({risk_category(r['p'])[0]}), health score "
                 f"{r['score']}/100. Top driver: {FACTOR_TEXT[r['factors'][0][0]][0 if r['factors'][0][1] > 0 else 1].lower()}.")
    else:
        for keys, ans in KB:
            if any(k in m for k in keys):
                reply = ans; break
    reply = reply or "I can help with glucose, HbA1c, BMI, diet, exercise, sleep, stress, symptoms and your latest result. Try “What is my risk?”."
    hist = (hist or []) + [{"role": "user", "content": msg}, {"role": "assistant", "content": reply}]
    return "", hist


# =============================================================================
# 10. USER INTERFACE
# =============================================================================
HEAD = """<link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/css/bootstrap.min.css" rel="stylesheet">
<link href="https://fonts.googleapis.com/css2?family=Instrument+Sans:wght@400;500;600;700&display=swap" rel="stylesheet">"""
CSS = """
:root{--dv-ink:#0F2A3A;--dv-teal:#0E8A8A;--dv-wash:#EAF5F3;--dv-card:#FFFFFF;--dv-line:#D3E3E0;--dv-mute:#587280}
.dark{--dv-ink:#E6F1F0;--dv-teal:#4FD1C5;--dv-wash:#10252F;--dv-card:#142F3B;--dv-line:#27444F;--dv-mute:#94ABB6}
body,.gradio-container{font-family:'Instrument Sans',system-ui,sans-serif!important;background:var(--body-background-fill)!important;color:var(--body-text-color)!important}
.dv-card{background:var(--dv-card);border:1px solid var(--dv-line);border-radius:10px;padding:14px 16px;height:100%;color:var(--dv-ink)}
.dv-card h5{color:var(--dv-ink);margin-bottom:10px}.dv-card .table{--bs-table-bg:transparent;--bs-table-color:var(--dv-ink)}
.dv-k{font-size:.8rem;color:var(--dv-mute);margin-bottom:2px}.dv-v{font-size:1.5rem;font-weight:700;line-height:1.2}
.dv-head{background:linear-gradient(120deg,#0F2A3A,#0E6F78);color:#fff;border-radius:12px;padding:22px 26px}
.dv-head h1{margin:0;font-weight:700;font-size:2rem;color:#fff}.dv-head p{margin:2px 0 0;opacity:.9;color:#fff}
.dv-needle{transform-origin:150px 150px;animation:dvSweep 1.8s cubic-bezier(.22,1,.36,1) both}
@keyframes dvSweep{from{transform:rotate(-90deg)}to{transform:rotate(var(--to))}}
.dv-bar{animation:dvGrow 1.2s ease-out both}@keyframes dvGrow{from{transform:scaleX(0);transform-origin:left}to{transform:scaleX(1);transform-origin:left}}
.dv-note{font-size:.82rem;color:var(--dv-mute)}
@media (prefers-reduced-motion:reduce){.dv-needle{animation:none;transform:rotate(var(--to))}.dv-bar{animation:none}}
"""
theme = gr.themes.Soft(primary_hue="teal", neutral_hue="slate", font=["Instrument Sans", "system-ui", "sans-serif"])
WAKE = [f"{h:02d}:{m:02d}" for h in range(4, 10) for m in (0, 30)]

HEADER = """<div class="dv-head"><h1>🩺 Diavision</h1><p><b>See Your Health. Predict Your Future.</b></p>
<p style="font-size:.9rem">Powered by Artificial Intelligence for Early Diabetes Detection and Smarter Healthcare.</p></div>"""


def banner_html():
    return f"""<div class="row g-2">
<div class="col-12 col-lg-6"><div class="dv-card"><div class="dv-k">AI Model Selection Engine</div>
<div class="dv-v">🏆 Best AI Model Selected: {BEST_NAME}</div>
<div class="dv-note">Chosen automatically for the highest validation ROC-AUC (F1 breaks ties) among {len(RES)} models.</div></div></div>
<div class="col-4 col-lg-2"><div class="dv-card"><div class="dv-k">Final accuracy (test)</div><div class="dv-v">{BEST_ROW['Test Accuracy']*100:.1f}%</div></div></div>
<div class="col-4 col-lg-2"><div class="dv-card"><div class="dv-k">Confidence score</div><div class="dv-v">{MEAN_CONF*100:.1f}%</div></div></div>
<div class="col-4 col-lg-2"><div class="dv-card"><div class="dv-k">ROC-AUC (test)</div><div class="dv-v">{BEST_ROW['Test ROC-AUC']:.3f}</div></div></div></div>
<p class="dv-note mt-2">Data source: <b>{DATA_SOURCE}</b>, {len(DF):,} records. Metrics are measured on a held-out test set of this dataset
and are not real-world clinical performance. Confidence score = mean of max(p, 1−p) over test predictions.</p>"""


with gr.Blocks(title="Diavision – AI Diabetes Prediction", theme=theme, css=CSS, head=HEAD) as demo:
    state = gr.State(None)
    gr.HTML(HEADER)
    with gr.Row():
        theme_btn = gr.Button("🌓 Dark / Light mode", size="sm", scale=0)
    theme_btn.click(None, None, None, js="() => {document.body.classList.toggle('dark')}")

    with gr.Tabs():
        # ---------------------------------------------------------------- Predict
        with gr.Tab("🔮 Predict"):
            with gr.Row():
                with gr.Column(scale=4, min_width=320):
                    gr.Markdown("### Patient details")
                    name = gr.Textbox(label="Full name", placeholder="e.g. Asha Patil")
                    email = gr.Textbox(label="Email (optional)", placeholder="asha@example.com")
                    with gr.Row():
                        age = gr.Number(label="Age", value=45, precision=0)
                        gender = gr.Radio(["Female", "Male"], label="Gender", value="Female")
                    with gr.Row():
                        height = gr.Number(label="Height (cm)", value=165)
                        weight = gr.Number(label="Weight (kg)", value=78)
                    with gr.Row():
                        glucose = gr.Number(label="Glucose (mg/dL)", value=128)
                        hba1c = gr.Number(label="HbA1c (%)", value=6.2)
                    with gr.Row():
                        bp = gr.Number(label="Blood pressure (systolic)", value=128)
                        insulin = gr.Number(label="Insulin (µU/mL)", value=70)
                    family = gr.Radio(["No", "Yes"], label="Family history of diabetes", value="Yes")
                    activity = gr.Slider(0, 20, value=2, step=.5, label="Physical activity (hours/week)")
                    sleep = gr.Slider(3, 11, value=6, step=.5, label="Sleep (hours/night)")
                    wake = gr.Dropdown(WAKE, value="06:30", label="Usual wake-up time")
                    go_btn = gr.Button("Run AI prediction", variant="primary", size="lg")
                with gr.Column(scale=7):
                    out_sum = gr.HTML("<div class='dv-card'>Enter patient details and press <b>Run AI prediction</b>.</div>")
                    out_meter = gr.HTML()
                    out_score = gr.HTML()
                    out_xai = gr.HTML()
            out_time = gr.Plot(label="Diabetes risk timeline")
            gr.Markdown("### AI lifestyle recommendations")
            out_recs = gr.HTML()
            with gr.Row():
                pdf_btn = gr.Button("🖨️ Create printable PDF report")
                pdf_file = gr.File(label="PDF report")
            go_btn.click(run_prediction, [name, email, age, gender, height, weight, glucose, hba1c, bp, insulin, family, activity, sleep, wake],
                         [out_sum, out_meter, out_xai, out_score, out_recs, out_time, state])
            pdf_btn.click(make_pdf, state, pdf_file)

        # -------------------------------------------------------------- Model Lab
        with gr.Tab("🧠 AI Model Lab"):
            gr.HTML(banner_html())
            gr.Markdown("### Model comparison (validation metrics used for selection)")
            gr.Dataframe(comparison_table(), interactive=False, wrap=True)
            gr.Plot(fig_model_bars())

        # -------------------------------------------------------------- Analytics
        with gr.Tab("📊 Analytics"):
            gr.HTML(f"<div class='row g-2'><div class='col-6 col-md-3'><div class='dv-card'><div class='dv-k'>Overall model accuracy (test)</div>"
                    f"<div class='dv-v'>{BEST_ROW['Test Accuracy']*100:.1f}%</div></div></div>"
                    f"<div class='col-6 col-md-3'><div class='dv-card'><div class='dv-k'>Records</div><div class='dv-v'>{len(DF):,}</div></div></div>"
                    f"<div class='col-6 col-md-3'><div class='dv-card'><div class='dv-k'>Diabetes prevalence</div><div class='dv-v'>{DF['Outcome'].mean()*100:.1f}%</div></div></div>"
                    f"<div class='col-6 col-md-3'><div class='dv-card'><div class='dv-k'>Best model</div><div class='dv-v'>{BEST_NAME}</div></div></div></div>")
            with gr.Row():
                gr.Plot(fig_cm()); gr.Plot(fig_roc())
            with gr.Row():
                gr.Plot(fig_importance()); gr.Plot(fig_shap())
            gr.Plot(fig_corr())
            with gr.Row():
                gr.Plot(fig_risk_dist()); trend = gr.Plot(fig_trend())
            with gr.Row():
                gr.Plot(fig_hist("Age", "Age distribution")); gr.Plot(fig_hist("BMI", "BMI distribution"))
            gr.Markdown("### Diabetes vs non-diabetes statistics")
            gr.Dataframe(stats_table(), interactive=False)
            gr.Button("Refresh trends").click(fig_trend, None, trend)
            demo.load(fig_trend, None, trend)

        # --------------------------------------------------------------- Patients
        with gr.Tab("🗂️ Patient Records"):
            q = gr.Textbox(label="Search by Patient ID, name or email", placeholder="DV000001, Asha, asha@example.com")
            with gr.Row():
                s_btn = gr.Button("Search", variant="primary"); csv_btn = gr.Button("⬇️ Export CSV")
            tbl = gr.Dataframe(history_df(), interactive=False, wrap=True)
            csv_file = gr.File(label="CSV export")
            s_btn.click(history_df, q, tbl); q.submit(history_df, q, tbl); csv_btn.click(export_csv, None, csv_file)
            demo.load(history_df, q, tbl)

        # ------------------------------------------------------------------ Tools
        with gr.Tab("🧮 Health Tools"):
            with gr.Row():
                t_age = gr.Number(label="Age", value=35, precision=0); t_g = gr.Radio(["Female", "Male"], value="Female", label="Gender")
                t_h = gr.Number(label="Height (cm)", value=165); t_w = gr.Number(label="Weight (kg)", value=70)
                t_a = gr.Slider(0, 20, value=3, step=.5, label="Activity (hours/week)")
            tool_out = gr.HTML()

            def tools(a, g, h, w, act):
                b = bmi_calc(w, h); cat = "Underweight" if b < 18.5 else "Healthy" if b < 25 else "Overweight" if b < 30 else "Obese"
                bmr, tdee = calorie_calc(w, h, a, g, act); iw = ideal_weight(h, g)
                cards = [("BMI", f"{b:.1f} · {cat}"), ("Ideal weight", f"{iw:.1f} kg (Devine)"), ("Daily water", f"{water_litres(w, act)} L"),
                         ("Daily calories (maintenance)", f"{tdee} kcal"), ("Basal metabolic rate", f"{bmr} kcal")]
                return "<div class='row g-2'>" + "".join(f"<div class='col-6 col-md-4'><div class='dv-card'><div class='dv-k'>{k}</div><div class='dv-v' style='font-size:1.2rem'>{v}</div></div></div>" for k, v in cards) + "</div>"
            gr.Button("Calculate", variant="primary").click(tools, [t_age, t_g, t_h, t_w, t_a], tool_out)

        # -------------------------------------------------------------- Assistant
        with gr.Tab("💬 AI Health Assistant"):
            gr.Markdown("Ask about glucose, HbA1c, BMI, diet, exercise, sleep, stress or your latest result. "
                        "_This assistant uses built-in medical guidance rules (no external AI service) and does not replace a doctor._")
            chat = gr.Chatbot(type="messages", height=380)
            msg = gr.Textbox(placeholder="Ask a question and press Enter…", show_label=False)
            msg.submit(assistant, [msg, chat, state], [msg, chat])

    gr.HTML("<p class='dv-note text-center mt-3'>Diavision is a screening aid, not a medical diagnosis. Always consult a qualified healthcare professional.</p>")

if os.environ.get("DV_NO_LAUNCH") != "1":
    demo.queue().launch(share=True, debug=False)


Training models …
  Logistic Regression  val acc 0.822  val auc 0.899  (0.1s)
  Decision Tree        val acc 0.788  val auc 0.849  (0.1s)
  Random Forest        val acc 0.809  val auc 0.884  (3.9s)
  SVM                  val acc 0.819  val auc 0.874  (5.6s)
  XGBoost              val acc 0.809  val auc 0.887  (0.3s)
  LightGBM             val acc 0.810  val auc 0.884  (0.3s)
  CatBoost             val acc 0.811  val auc 0.891  (0.6s)
Best model: Logistic Regression  | test acc 0.826 | mean confidence 0.822


ExactExplainer explainer: 251it [00:11, 10.46it/s]                         


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8e3eef88bdd06fd729.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
